In [29]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
from scipy.ndimage import gaussian_filter

In [34]:
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*blos_20260[4-5]*'))[:220]
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_Q/*'))[-27:]
files

['/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260617T020009_V202609151150C_0646170501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260618T020009_V202609151151C_0646180501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260619T020009_V202609151151C_0646190501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260620T020009_V202609151151C_0646200501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260623T050009_V202609151152C_0646230501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260624T030009_V202609151152C_0646240501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260625T030009_V202609151152C_0646250501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260626T053009_V202609151153C_0646260501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260627T030009_V202609151153C_0646270501.fits',
 '/home/ulyanov/data/solo/phi/2026/blos_Q/phi-fdt-blos_20260628T

In [35]:
data, alpha = make_map(files, correct_dr=True, sine_lat=True, correct_bias=False, minmu=0.1, delta_lon=90, pow=2, lam=0.05, binning=1)

In [36]:
show_map(data, vmin=-1000, vmax=1000, sine_lat=True)

In [37]:
nx = len(data)
#lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
#flux = np.nanmean(data, axis=1)
flux = np.nanmean(data * alpha, axis=1) / np.nanmean(alpha, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_21483/1563973450.py:5: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(data * alpha, axis=1) / np.nanmean(alpha, axis=1)


In [6]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2310.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.04.29_07:08:43_TAI'

In [7]:
show_map(data_, vmin=-1000, vmax=1000, sine_lat=True)

In [8]:
#flux_ = np.nanmean(data_ * alpha, axis=1) / np.nanmean(alpha, axis=1)
flux_ = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_ = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_, flux_)
plt.plot(lat, flux)


plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_131644/2474185863.py:2: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(data_, axis=1)


In [27]:
from modulation import *
from wavelengths import *

O = modulation_matrix()
D = np.linalg.inv(O)

In [37]:
voltages = []

for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header_data = hdul[0].header
        img_data = hdul['PHI_FITS_imageSummary'].data
        fg_data = hdul['PHI_FITS_FG_settings'].data
        fpa_data = hdul['PHI_FITS_FPA_settings'].data
        pmp_data = hdul['PHI_FITS_PMP_settings'].data

    V = get_mean_voltages(header_data, fg_data, pmp_data)
    voltages += [D @ V]

voltages = np.array(voltages)

In [48]:
with fits.open(files[1]) as hdul:
    data = hdul[0].data
    header_data = hdul[0].header
    img_data = hdul['PHI_FITS_imageSummary'].data
    fg_data = hdul['PHI_FITS_FG_settings'].data
    fpa_data = hdul['PHI_FITS_FPA_settings'].data
    pmp_data = hdul['PHI_FITS_PMP_settings'].data

In [49]:
fpa_data

FITS_rec([('2026-04-09 08:00:02', '2026-04-09T08:00:03.107', 'FITS_FPA_SET', np.int32(8), np.int32(256), np.int32(766), np.int32(4), np.int64(1775721260), np.int64(203341), np.int64(1), np.float64(0.013)),
          ('2026-04-09 08:00:02', '2026-04-09T08:00:03.542', 'FITS_FPA_SET', np.int32(8), np.int32(256), np.int32(766), np.int32(4), np.int64(1775721260), np.int64(638351), np.int64(1), np.float64(0.013)),
          ('2026-04-09 08:00:03', '2026-04-09T08:00:04.022', 'FITS_FPA_SET', np.int32(8), np.int32(256), np.int32(766), np.int32(4), np.int64(1775721261), np.int64(118498), np.int64(1), np.float64(0.013)),
          ('2026-04-09 08:00:03', '2026-04-09T08:00:04.497', 'FITS_FPA_SET', np.int32(8), np.int32(256), np.int32(766), np.int32(4), np.int64(1775721261), np.int64(593633), np.int64(1), np.float64(0.013)),
          ('2026-04-09 08:00:04', '2026-04-09T08:00:04.972', 'FITS_FPA_SET', np.int32(8), np.int32(256), np.int32(766), np.int32(4), np.int64(1775721262), np.int64(68342), np.i

In [50]:
fg_data

FITS_rec([('2026-04-09 08:00:02', 'FITS_FG_SET', np.int32(-455), np.int64(6173181), np.int64(351300), np.int64(6173341), np.int64(0), np.float64(6173.381)),
          ('2026-04-09 08:00:03', 'FITS_FG_SET', np.int32(-454), np.int64(6173181), np.int64(351300), np.int64(6173341), np.int64(0), np.float64(6173.381)),
          ('2026-04-09 08:00:04', 'FITS_FG_SET', np.int32(-455), np.int64(6173181), np.int64(351300), np.int64(6173341), np.int64(0), np.float64(6173.381)),
          ('2026-04-09 08:00:04', 'FITS_FG_SET', np.int32(-454), np.int64(6173181), np.int64(351300), np.int64(6173341), np.int64(0), np.float64(6173.381)),
          ('2026-04-09 08:00:05', 'FITS_FG_SET', np.int32(-455), np.int64(6173181), np.int64(351300), np.int64(6173341), np.int64(0), np.float64(6173.381)),
          ('2026-04-09 08:00:05', 'FITS_FG_SET', np.int32(-454), np.int64(6173181), np.int64(351300), np.int64(6173341), np.int64(0), np.float64(6173.381)),
          ('2026-04-09 08:00:05', 'FITS_FG_SET', np.int32(

In [51]:
pmp_data

FITS_rec([('2026-04-09 08:00:02', 'FITS_PMP_SET', np.int16(1), np.int32(2074), np.int32(2030), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:02', 'FITS_PMP_SET', np.int16(2), np.int32(2074), np.int32(2986), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:03', 'FITS_PMP_SET', np.int16(3), np.int32(1682), np.int32(5922), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:03', 'FITS_PMP_SET', np.int16(4), np.int32(1682), np.int32(1725), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:04', 'FITS_PMP_SET', np.int16(1), np.int32(2074), np.int32(2030), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:04', 'FITS_PMP_SET', np.int16(2), np.int32(2074), np.int32(2986), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:05', 'FITS_PMP_SET', np.int16(3), np.int32(1682), np.int32(5922), np.int16(255), np.int32(0), np.int32(0)),
          ('2026-04-09 08:00:05', 'FITS_P

In [29]:
q = np.mean(voltages, axis=0)

In [30]:
q

array([[-5.32985603e+02, -8.27022393e+01,  1.13847585e+02,
         3.10412071e+02,  5.08451873e+02,  7.06610986e+02],
       [-4.60593959e-01, -7.62430899e-02,  1.00541767e-01,
         2.92721639e-01,  4.40055343e-01,  6.21756013e-01],
       [-2.75039825e+00, -4.31063903e-01,  5.56976015e-01,
         1.57136497e+00,  2.62175202e+00,  3.60344033e+00],
       [ 1.32250674e+00,  1.84271920e-01, -3.02286053e-01,
        -7.69736338e-01, -1.27920359e+00, -1.77853877e+00]])